# Multi-agent Systems and Agentic Patterns with CrewAI

## Introduction

**CrewAI** is a powerful framework for orchestrating role-playing AI agents that work together to accomplish complex tasks. It provides a simple yet flexible API for creating multi-agent systems where each agent has specific roles, goals, and capabilities.

In this tutorial, we'll explore five fundamental agentic patterns using CrewAI with an **Azure OpenAI** deployment (`gpt-5.4-mini` or `gpt-5.6-luna`) as the model backend.

## 1. Setup & Installation

First, let's install the required libraries.

> **Note:** CrewAI supports Python 3.10–3.13 and pins many of its own dependency versions, so run this notebook in a fresh environment (for example a new Colab runtime or a dedicated virtual environment).

In [1]:
# Install required packages
# crewai: The main framework for multi-agent orchestration
# python-dotenv: Loads credentials from the .env file
# CrewAI supports Python 3.10-3.13

%pip install -q "crewai>=1.15" python-dotenv

# Windows only: pywin32 (used by the MCP SDK) normally needs a kernel restart after installation.
# Re-reading the site-packages .pth files makes it importable right away.
import site, sys, sysconfig
if sys.platform == "win32":
    for _site_dir in (sysconfig.get_paths()["purelib"], site.getusersitepackages()):
        site.addsitedir(_site_dir)

Note: you may need to restart the kernel to use updated packages.


### Configure Azure OpenAI Credentials

This notebook uses an **Azure OpenAI** deployment through its OpenAI-compatible **v1 endpoint**.
Credentials are read from a `.env` file in this notebook's folder (or any parent folder). Copy `example.env` to `.env` and fill it in. On **Google Colab**, upload `.env` with the 📁 *Files* panel, or just enter the values when prompted:

| Variable | Description |
|----------|-------------|
| `AZURE_OPENAI_API_KEY` | Your Azure OpenAI API key |
| `AZURE_OPENAI_ENDPOINT` | Base URL, e.g. `https://<resource>.openai.azure.com/` (the full `.../openai/v1/` URL also works) |
| `AZURE_OPENAI_DEPLOYMENT` | Deployment (model) name: `gpt-5.4-mini` or `gpt-5.6-luna` |

If a value is missing, you'll be prompted for it.

**Security Best Practice**: Never hardcode API keys in your code. Keep them in `.env` (which must never be committed or shared).

In [2]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


## 2. Model Configuration

Let's create a reusable LLM configuration using CrewAI's `LLM` class, pointed at our Azure OpenAI deployment. The `openai/` prefix selects CrewAI's native OpenAI provider, and `base_url` points it at the Azure OpenAI v1 endpoint.

In [3]:
from crewai import LLM

llm = LLM(
    model=f"openai/{AZURE_OPENAI_DEPLOYMENT}",  # Prefix with openai/ so it uses the OpenAI-compatible protocol
    base_url=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

print(f"✅ Language model configured: {AZURE_OPENAI_DEPLOYMENT} (Azure OpenAI)")

✅ Language model configured: gpt-5.4-mini (Azure OpenAI)


## 3. Pattern A: Prompt Chaining

**Concept**: In prompt chaining, the output of one agent becomes the input for the next agent, creating a sequential pipeline of processing.

**Use Case**: When a task naturally breaks into sequential steps where each step depends on the previous one.

**Example**: Research → Summarization

> **Note:** Jupyter already runs an asyncio event loop, and CrewAI refuses to start a crew synchronously inside a running loop. In notebooks we therefore use `await crew.kickoff_async(...)`; in a plain Python script you can call `crew.kickoff(...)` instead.

In [4]:
from crewai import Agent, Task, Crew

# Agent A: Research Agent
# This agent gathers detailed information on a topic
research_agent = Agent(
    role="Research Specialist",
    goal="Gather comprehensive information about {topic}",
    backstory="You are an expert researcher with deep knowledge across multiple domains. "
              "You excel at finding relevant facts and insights.",
    llm=llm,
    verbose=True
)

# Agent B: Summary Agent
# This agent takes the research output and creates a concise summary
summary_agent = Agent(
    role="Content Summarizer",
    goal="Create a clear and concise summary of the research findings",
    backstory="You are a skilled writer who excels at distilling complex information "
              "into easily digestible summaries.",
    llm=llm,
    verbose=True
)

# Task 1: Research Task (Agent A)
# Output will be passed to the next task
research_task = Task(
    description="Research and gather detailed information about {topic}. "
                "Include key facts, recent developments, and important context.",
    agent=research_agent,
    expected_output="A comprehensive research document with key facts and insights"
)

# Task 2: Summary Task (Agent B)
# This task receives the output from research_task as context
summary_task = Task(
    description="Take the research findings and create a concise 3-paragraph summary. "
                "Focus on the most important points.",
    agent=summary_agent,
    expected_output="A clear 3-paragraph summary of the research findings",
    context=[research_task]  # This creates the chain: research_task output → summary_task input
)

# Create the crew with sequential process (default)
chain_crew = Crew(
    agents=[research_agent, summary_agent],
    tasks=[research_task, summary_task],
    verbose=True
)

# Execute the chain
print("=" * 60)
print("PATTERN A: PROMPT CHAINING EXAMPLE")
print("=" * 60)

result = await chain_crew.kickoff_async(inputs={"topic": "Quantum Computing"})
print("\n" + "=" * 60)
print("FINAL RESULT:")
print("=" * 60)
print(result)

PATTERN A: PROMPT CHAINING EXAMPLE


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.44                                                                                       │
│  Latest version:  1.15.45                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 153e32ef-f58e-4a44-8291-5e87eb1669f0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research and gather detailed information about Quantum Computing. Include key facts, recent              │
│  developments, and important context.                                                                           │
│  ID: 983dfa36-bfdc-4144-ab87-47bfff9e0f96                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task: Research and gather detailed information about Quantum Computing. Include key facts, recent              │
│  developments, and important context.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Using an approximate input token count for model 'gpt-5.4-mini': assuming one token per four characters because a native token count is unavailable.


... (6 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Summarizer                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Quantum computing is an emerging computing paradigm that uses qubits, superposition, entanglement, and         │
│  interference to solve certain problems more efficiently than classical computers. The strongest theoretical    │
│  advantages are in areas such as integer factoring, search, and especially quantum simulation, which could      │
│  have major implications for chemistry, materials science, and fundamental physics. However, today’s systems    │
│  are still in the noisy, pre-fault-tolerant NISQ era, meaning they are limited by decoherence, gate errors,     │
│  and scaling challenges.                                                                                        │
│                                                                                                                 │
│  The central technical barrier is error correction. Useful large-scale quantum computing will require many      │
│  high-quality physical qubits to create a small number of stable logical qubits, with the surface code          │
│  currently being one of the most promising approaches. Recent progress has been encouraging, including better   │
│  fidelities, repeated error-correction cycles, and early demonstrations of logical qubits with improved error   │
│  suppression. Still, current hardware cannot yet deliver broad practical advantage over classical               │
│  supercomputers for most workloads, and significant engineering breakthroughs are still needed.                 │
│                                                                                                                 │
│  Multiple hardware platforms are advancing, including superconducting qubits, trapped ions, neutral atoms,      │
│  photonics, and semiconductor spin qubits, each with different strengths and tradeoffs. Commercial investment   │
│  and national programs are strong, and the industry is increasingly focused on realistic near-term uses such    │
│  as chemistry, materials, and cybersecurity rather than broad hype. One of the most urgent real-world impacts   │
│  is cryptography: sufficiently powerful fault-tolerant quantum computers could break RSA and ECC, making        │
│  post-quantum cryptography migration a pressing priority.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Take the research findings and create a concise 3-paragraph summary. Focus on the most important         │
│  points.                                                                                                        │
│  Agent: Content Summarizer                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 153e32ef-f58e-4a44-8291-5e87eb1669f0                                                                       │
│  Final Output: Quantum computing is an emerging computing paradigm that uses qubits, superposition,             │
│  entanglement, and interference to solve certain problems more efficiently than classical computers. The        │
│  strongest theoretical advantages are in areas such as integer factoring, search, and especially quantum        │
│  simulation, which could have major implications for chemistry, materials science, and fundamental physics.     │
│  However, today’s systems are still in the noisy, pre-fault-tolerant NISQ era, meaning they are limited by      │
│  decoherence, gate errors, and scaling challenges.                                                              │
│                                                                                                                 │
│  The central technical barrier is error correction. Useful large-scale quantum computing will require many      │
│  high-quality physical qubits to create a small number of stable logical qubits, with the surface code          │
│  currently being one of the most promising approaches. Recent progress has been encouraging, including better   │
│  fidelities, repeated error-correction cycles, and early demonstrations of logical qubits with improved error   │
│  suppression. Still, current hardware cannot yet deliver broad practical advantage over classical               │
│  supercomputers for most workloads, and significant engineering breakthroughs are still needed.                 │
│                                                                                                                 │
│  Multiple hardware platforms are advancing, including superconducting qubits, trapped ions, neutral atoms,      │
│  photonics, and semiconductor spin qubits, each with different strengths and tradeoffs. Commercial investment   │
│  and national programs are strong, and the industry is increasingly focused on realistic near-term uses such    │
│  as chemistry, materials, and cybersecurity rather than broad hype. One of the most urgent real-world impacts   │
│  is cryptography: sufficiently powerful fault-tolerant quantum computers could break RSA and ECC, making        │
│  post-quantum cryptography migration a pressing priority.                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL RESULT:
Quantum computing is an emerging computing paradigm that uses qubits, superposition, entanglement, and interference to solve certain problems more efficiently than classical computers. The strongest theoretical advantages are in areas such as integer factoring, search, and especially quantum simulation, which could have major implications for chemistry, materials science, and fundamental physics. However, today’s systems are still in the noisy, pre-fault-tolerant NISQ era, meaning they are limited by decoherence, gate errors, and scaling challenges.

The central technical barrier is error correction. Useful large-scale quantum computing will require many high-quality physical qubits to create a small number of stable logical qubits, with the surface code currently being one of the most promising approaches. Recent progress has been encouraging, including better fidelities, repeated error-correction cycles, and early demonstrations of logical qubits with improved error su

## 4. Pattern B: Routing

**Concept**: A router agent analyzes incoming requests and directs them to the appropriate specialized agent based on content or intent.

**Use Case**: When you have multiple specialized agents and need intelligent request distribution.

**Example**: Customer Service Router → Specialized Support Agents

In [5]:
# Specialized Agent 1: Technical Support
technical_agent = Agent(
    role="Technical Support Specialist",
    goal="Resolve technical issues and provide troubleshooting guidance",
    backstory="You are a technical expert with deep knowledge of software, hardware, "
              "and IT systems. You excel at diagnosing and solving technical problems.",
    llm=llm,
    verbose=True
)

# Specialized Agent 2: Sales Agent
sales_agent = Agent(
    role="Sales Representative",
    goal="Provide product information and help customers make purchase decisions",
    backstory="You are a knowledgeable sales professional who understands customer needs "
              "and can recommend the right products and services.",
    llm=llm,
    verbose=True
)

# Specialized Agent 3: General Information Agent
general_agent = Agent(
    role="General Information Specialist",
    goal="Answer general questions about company policies, hours, and basic information",
    backstory="You are a friendly customer service representative with comprehensive "
              "knowledge of company policies and general information.",
    llm=llm,
    verbose=True
)

# Router Agent: Analyzes queries and routes to appropriate specialist
router_agent = Agent(
    role="Customer Service Router",
    goal="Analyze customer queries and determine which specialist should handle them",
    backstory="You are an intelligent routing system that can quickly categorize customer "
              "inquiries as: TECHNICAL (tech issues), SALES (product/pricing questions), "
              "or GENERAL (policies/hours/basic info).",
    llm=llm,
    verbose=True
)

# Router Task: Determines which specialist to use
routing_task = Task(
    description="Analyze this customer query: '{query}'\n\n"
                "Determine if this is a TECHNICAL, SALES, or GENERAL inquiry. "
                "Respond with ONLY one word: TECHNICAL, SALES, or GENERAL.",
    agent=router_agent,
    expected_output="One word: TECHNICAL, SALES, or GENERAL"
)

# Create tasks for each specialist
technical_task = Task(
    description="Handle this technical inquiry: '{query}'",
    agent=technical_agent,
    expected_output="Technical support response with solution steps"
)

sales_task = Task(
    description="Handle this sales inquiry: '{query}'",
    agent=sales_agent,
    expected_output="Sales response with product recommendations"
)

general_task = Task(
    description="Handle this general inquiry: '{query}'",
    agent=general_agent,
    expected_output="General information response"
)

# Function to route queries
async def route_query(query):
    """Route a query to the appropriate specialist agent"""
    print("=" * 60)
    print(f"ROUTING QUERY: {query}")
    print("=" * 60)

    # First, determine the route
    route_crew = Crew(
        agents=[router_agent],
        tasks=[routing_task],
        verbose=True
    )

    route_result = await route_crew.kickoff_async(inputs={"query": query})
    route = str(route_result).strip().upper()

    print(f"\n→ Route determined: {route}\n")

    # Then, execute the appropriate specialist task
    if "TECHNICAL" in route:
        specialist_crew = Crew(agents=[technical_agent], tasks=[technical_task], verbose=True)
    elif "SALES" in route:
        specialist_crew = Crew(agents=[sales_agent], tasks=[sales_task], verbose=True)
    else:
        specialist_crew = Crew(agents=[general_agent], tasks=[general_task], verbose=True)

    result = await specialist_crew.kickoff_async(inputs={"query": query})

    print("\n" + "=" * 60)
    print("FINAL RESPONSE:")
    print("=" * 60)
    print(result)
    return result

# Test the routing pattern
print("\n\n" + "=" * 60)
print("PATTERN B: ROUTING EXAMPLE")
print("=" * 60 + "\n")

# Test with a technical query
await route_query("My laptop won't connect to WiFi. How do I fix this?")



PATTERN B: ROUTING EXAMPLE

ROUTING QUERY: My laptop won't connect to WiFi. How do I fix this?


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.44                                                                                       │
│  Latest version:  1.15.45                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 67fad0f3-4937-46e1-afdb-e9eb4d86f471                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze this customer query: 'My laptop won't connect to WiFi. How do I fix this?'                       │
│                                                                                                                 │
│  Determine if this is a TECHNICAL, SALES, or GENERAL inquiry. Respond with ONLY one word: TECHNICAL, SALES, or  │
│  GENERAL.                                                                                                       │
│  ID: 8509053c-9753-43e1-9113-5384b263a8e8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Service Router                                                                                 │
│                                                                                                                 │
│  Task: Analyze this customer query: 'My laptop won't connect to WiFi. How do I fix this?'                       │
│                                                                                                                 │
│  Determine if this is a TECHNICAL, SALES, or GENERAL inquiry. Respond with ONLY one word: TECHNICAL, SALES, or  │
│  GENERAL.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (10 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Handle this technical inquiry: 'My laptop won't connect to WiFi. How do I fix this?'                     │
│  Agent: Technical Support Specialist                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL RESPONSE:
Here are the most effective steps to fix a laptop that won’t connect to Wi‑Fi. Try them in order:

## 1) Check the basics
- Make sure **Wi‑Fi is turned on** on the laptop.
- Confirm **Airplane mode is off**.
- Check whether **other devices can connect** to the same Wi‑Fi network.
  - If nothing can connect, the issue is likely with the **router or internet service**, not the laptop.

## 2) Restart everything
- **Restart your laptop**.
- **Power cycle your router/modem**:
  1. Unplug the router and modem from power.
  2. Wait **30 seconds**.
  3. Plug the modem back in and wait until it fully starts.
  4. Plug the router back in and wait for all lights to stabilize.

## 3) Forget and reconnect to the network
- Go to your Wi‑Fi settings.
- Select your network and choose **Forget** / **Remove**.
- Reconnect by entering the Wi‑Fi password again.

## 4) Check password and signal
- Make sure you’re entering the **correct password**.
- Move closer to the router to rule out we

CrewOutput(raw='Here are the most effective steps to fix a laptop that won’t connect to Wi‑Fi. Try them in order:\n\n## 1) Check the basics\n- Make sure **Wi‑Fi is turned on** on the laptop.\n- Confirm **Airplane mode is off**.\n- Check whether **other devices can connect** to the same Wi‑Fi network.\n  - If nothing can connect, the issue is likely with the **router or internet service**, not the laptop.\n\n## 2) Restart everything\n- **Restart your laptop**.\n- **Power cycle your router/modem**:\n  1. Unplug the router and modem from power.\n  2. Wait **30 seconds**.\n  3. Plug the modem back in and wait until it fully starts.\n  4. Plug the router back in and wait for all lights to stabilize.\n\n## 3) Forget and reconnect to the network\n- Go to your Wi‑Fi settings.\n- Select your network and choose **Forget** / **Remove**.\n- Reconnect by entering the Wi‑Fi password again.\n\n## 4) Check password and signal\n- Make sure you’re entering the **correct password**.\n- Move closer to the

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: bd0c3ad3-673a-4bce-8ea7-dda51f5e9723                                                                       │
│  Final Output: Here are the most effective steps to fix a laptop that won’t connect to Wi‑Fi. Try them in       │
│  order:                                                                                                         │
│                                                                                                                 │
│  ## 1) Check the basics                                                                                         │
│  - Make sure **Wi‑Fi is turned on** on the laptop.                                                              │
│  - Confirm **Airplane mode is off**.                                                                            │
│  - Check whether **other devices can connect** to the same Wi‑Fi network.                                       │
│    - If nothing can connect, the issue is likely with the **router or internet service**, not the laptop.       │
│                                                                                                                 │
│  ## 2) Restart everything                                                                                       │
│  - **Restart your laptop**.                                                                                     │
│  - **Power cycle your router/modem**:                                                                           │
│    1. Unplug the router and modem from power.                                                                   │
│    2. Wait **30 seconds**.                                                                                      │
│    3. Plug the modem back in and wait until it fully starts.                                                    │
│    4. Plug the router back in and wait for all lights to stabilize.                                             │
│                                                                                                                 │
│  ## 3) Forget and reconnect to the network                                                                      │
│  - Go to your Wi‑Fi settings.                                                                                   │
│  - Select your network and choose **Forget** / **Remove**.                                                      │
│  - Reconnect by entering the Wi‑Fi password again.                                                              │
│                                                                                                                 │
│  ## 4) Check password and signal                                                                                │
│  - Make sure you’re entering the **correct password**.                                                          │
│  - Move closer to the router to rule out weak signal.                                                           │
│  - If your router has both **2.4 GHz and 5 GHz**, try connecting to the other band.                             │
│                                                                                                                 │
│  ## 5) Disable and re-enable the Wi‑Fi adapter                                                                  │
│  - On Windows:                                                                                                  │
│    - Open **Device Manager** > **Network adapters**   

## 5. Pattern C: Parallelization

**Concept**: Multiple agents work simultaneously on different aspects of a problem, then their results are aggregated.

**Use Case**: When you have independent sub-tasks that can be executed concurrently to save time.

**Example**: Market Research + Competitor Analysis → Unified Report

**How**: Tasks marked with `async_execution=True` run concurrently. A later task that lists them in `context` waits for all of them to finish before it starts.

In [6]:
from crewai import Process

# Agent 1: Market Research Agent (runs in parallel)
market_researcher = Agent(
    role="Market Research Analyst",
    goal="Analyze market trends, customer needs, and opportunities for {product}",
    backstory="You are a market research expert with years of experience in analyzing "
              "market dynamics, consumer behavior, and industry trends.",
    llm=llm,
    verbose=True
)

# Agent 2: Competitor Analysis Agent (runs in parallel)
competitor_analyst = Agent(
    role="Competitive Intelligence Analyst",
    goal="Analyze competitors, their strategies, and market positioning for {product}",
    backstory="You are a competitive intelligence specialist who excels at understanding "
              "competitor strengths, weaknesses, and market strategies.",
    llm=llm,
    verbose=True
)

# Agent 3: Aggregator Agent (runs after parallel tasks complete)
aggregator = Agent(
    role="Strategic Report Writer",
    goal="Combine market research and competitor analysis into a unified strategic report",
    backstory="You are a strategic analyst who excels at synthesizing multiple sources "
              "of information into actionable insights.",
    llm=llm,
    verbose=True
)

# Parallel Task 1: Market Research
market_task = Task(
    description="Conduct market research for {product}. Analyze:\n"
                "1. Market size and growth potential\n"
                "2. Target customer segments\n"
                "3. Key market trends\n"
                "4. Opportunities and challenges",
    agent=market_researcher,
    expected_output="Comprehensive market research report with key insights",
    async_execution=True,  # Runs concurrently with competitor_task
)

# Parallel Task 2: Competitor Analysis
competitor_task = Task(
    description="Analyze competitors in the {product} market. Include:\n"
                "1. Main competitors\n"
                "2. Their strengths and weaknesses\n"
                "3. Market share and positioning\n"
                "4. Competitive advantages",
    agent=competitor_analyst,
    expected_output="Detailed competitor analysis report",
    async_execution=True,  # Runs concurrently with market_task
)

# Aggregation Task: Combines results from both parallel tasks
aggregation_task = Task(
    description="Synthesize the market research and competitor analysis into a unified "
                "strategic report. Provide actionable recommendations.",
    agent=aggregator,
    expected_output="Unified strategic report with recommendations",
    context=[market_task, competitor_task]  # Waits for both tasks to complete
)

# Create the crew: both async tasks start at the same time, and the
# aggregation task waits for their results (via `context`) before it runs
parallel_crew = Crew(
    agents=[market_researcher, competitor_analyst, aggregator],
    tasks=[market_task, competitor_task, aggregation_task],
    verbose=True,
    process=Process.sequential  # Tasks 1 & 2 run concurrently; Task 3 aggregates
)

# Execute parallel pattern
print("=" * 60)
print("PATTERN C: PARALLELIZATION EXAMPLE")
print("=" * 60)
print("Note: Market research and competitor analysis run concurrently (async_execution=True)")
print("=" * 60 + "\n")

result = await parallel_crew.kickoff_async(inputs={"product": "AI-powered fitness app"})

print("\n" + "=" * 60)
print("AGGREGATED RESULT:")
print("=" * 60)
print(result)

PATTERN C: PARALLELIZATION EXAMPLE
Note: Market research and competitor analysis run concurrently (async_execution=True)



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.44                                                                                       │
│  Latest version:  1.15.45                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1fa9a480-a44a-4c6a-8a1f-fb0f7242cf60                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Conduct market research for AI-powered fitness app. Analyze:                                             │
│  1. Market size and growth potential                                                                            │
│  2. Target customer segments                                                                                    │
│  3. Key market trends                                                                                           │
│  4. Opportunities and challenges                                                                                │
│  ID: bc76ee6f-fa8d-48e5-9ee2-08706dc4ae79                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze competitors in the AI-powered fitness app market. Include:                                       │
│  1. Main competitors                                                                                            │
│  2. Their strengths and weaknesses                                                                              │
│  3. Market share and positioning                                                                                │
│  4. Competitive advantages                                                                                      │
│  ID: be8d4cb8-bf74-4f2f-98a1-47e297a40cc1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Competitive Intelligence Analyst                                                                        │
│                                                                                                                 │
│  Task: Analyze competitors in the AI-powered fitness app market. Include:                                       │
│  1. Main competitors                                                                                            │
│  2. Their strengths and weaknesses                                                                              │
│  3. Market share and positioning                                                                                │
│  4. Competitive advantages                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (10 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Strategic Report Writer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Unified Strategic Report: AI-Powered Fitness App Market and Competitive Landscape                            │
│                                                                                                                 │
│  ## 1. Executive Summary                                                                                        │
│                                                                                                                 │
│  The AI-powered fitness app market is attractive, fast-growing, and highly competitive. It sits at the          │
│  intersection of digital fitness, personalized wellness, and AI-driven consumer applications, benefiting from   │
│  rising health awareness, subscription comfort, and increasing wearable adoption.                               │
│                                                                                                                 │
│  The market opportunity is strongest where AI can deliver measurable value: adaptive workout planning, form     │
│  and recovery guidance, progress prediction, nutrition support, and behavioral coaching. However, success will  │
│  not come from “AI” as a feature alone. The category is crowded with companies that already own one or more of  │
│  the critical advantages: large brand trust, device ecosystems, rich data, premium content, or human coaching.  │
│                                                                                                                 │
│  The strategic conclusion is clear:                                                                             │
│                                                                                                                 │
│  - **The market is large enough to support new entrants**                                                       │
│  - **Generic fitness apps will struggle**                                                                       │
│  - **Winning products must focus on a specific user segment**                                                   │
│  - **Differentiation must come from personalization quality, trust, and retention**                             │
│  - **A hybrid AI + human + wearable-enabled model is the strongest long-term opportunity**                      │
│                                                                                                                 │
│  The best entry strategy is to build a focused, outcome-driven fitness coach for a clearly defined audience     │
│  such as busy professionals, beginners, or weight loss users. The product should be positioned as a             │
│  personalized AI coach that adapts workouts, recovery, and habit support to the user’s schedule, goals, and     │
│  real-world constraints.                                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Market Attractiveness Assessment                                                                         │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Synthesize the market research and competitor analysis into a unified strategic report. Provide          │
│  actionable recommendations.                                                                                    │
│  Agent: Strategic Report Writer                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


AGGREGATED RESULT:
# Unified Strategic Report: AI-Powered Fitness App Market and Competitive Landscape

## 1. Executive Summary

The AI-powered fitness app market is attractive, fast-growing, and highly competitive. It sits at the intersection of digital fitness, personalized wellness, and AI-driven consumer applications, benefiting from rising health awareness, subscription comfort, and increasing wearable adoption.

The market opportunity is strongest where AI can deliver measurable value: adaptive workout planning, form and recovery guidance, progress prediction, nutrition support, and behavioral coaching. However, success will not come from “AI” as a feature alone. The category is crowded with companies that already own one or more of the critical advantages: large brand trust, device ecosystems, rich data, premium content, or human coaching.

The strategic conclusion is clear:

- **The market is large enough to support new entrants**
- **Generic fitness apps will struggle**
- **W

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1fa9a480-a44a-4c6a-8a1f-fb0f7242cf60                                                                       │
│  Final Output: # Unified Strategic Report: AI-Powered Fitness App Market and Competitive Landscape              │
│                                                                                                                 │
│  ## 1. Executive Summary                                                                                        │
│                                                                                                                 │
│  The AI-powered fitness app market is attractive, fast-growing, and highly competitive. It sits at the          │
│  intersection of digital fitness, personalized wellness, and AI-driven consumer applications, benefiting from   │
│  rising health awareness, subscription comfort, and increasing wearable adoption.                               │
│                                                                                                                 │
│  The market opportunity is strongest where AI can deliver measurable value: adaptive workout planning, form     │
│  and recovery guidance, progress prediction, nutrition support, and behavioral coaching. However, success will  │
│  not come from “AI” as a feature alone. The category is crowded with companies that already own one or more of  │
│  the critical advantages: large brand trust, device ecosystems, rich data, premium content, or human coaching.  │
│                                                                                                                 │
│  The strategic conclusion is clear:                                                                             │
│                                                                                                                 │
│  - **The market is large enough to support new entrants**                                                       │
│  - **Generic fitness apps will struggle**                                                                       │
│  - **Winning products must focus on a specific user segment**                                                   │
│  - **Differentiation must come from personalization quality, trust, and retention**                             │
│  - **A hybrid AI + human + wearable-enabled model is the strongest long-term opportunity**                      │
│                                                                                                                 │
│  The best entry strategy is to build a focused, outcome-driven fitness coach for a clearly defined audience     │
│  such as busy professionals, beginners, or weight loss users. The product should be positioned as a             │
│  personalized AI coach that adapts workouts, recovery, and habit support to the user’s schedule, goals, and     │
│  real-world constraints.                                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Market Attractiveness Assessment                                                                         │
│                                                       

## 6. Pattern D: Orchestrator-Worker

**Concept**: A manager/orchestrator agent breaks down complex tasks into subtasks and delegates them to worker agents sequentially.

**Use Case**: When a complex task needs to be decomposed into smaller, manageable steps that must be executed in order.

**Example**: Manager Agent → Worker Agent (Research → Outline → Draft → Edit)

In [7]:
# Manager Agent: Orchestrates the overall process
manager_agent = Agent(
    role="Project Manager",
    goal="Break down the task of writing a blog post about {topic} into clear subtasks",
    backstory="You are an experienced project manager who excels at decomposing complex "
              "tasks into manageable steps and ensuring quality execution.",
    llm=llm,
    verbose=True
)

# Worker Agent: Executes the subtasks
worker_agent = Agent(
    role="Content Worker",
    goal="Execute assigned subtasks efficiently and effectively",
    backstory="You are a versatile content professional capable of research, writing, "
              "and editing. You follow instructions precisely and deliver quality work.",
    llm=llm,
    verbose=True
)

# Step 1: Manager plans the work
planning_task = Task(
    description="Create a plan for writing a blog post about {topic}. "
                "List the key steps needed (research, outline, draft, edit).",
    agent=manager_agent,
    expected_output="A clear plan with 4 steps for creating the blog post"
)

# Step 2: Worker conducts research
research_step = Task(
    description="SUBTASK 1: Research {topic}. Gather key facts, statistics, and insights "
                "that would be valuable for a blog post.",
    agent=worker_agent,
    expected_output="Research findings with key facts and insights",
    context=[planning_task]
)

# Step 3: Worker creates outline
outline_step = Task(
    description="SUBTASK 2: Based on the research, create a detailed outline for a blog post "
                "about {topic}. Include main sections and key points.",
    agent=worker_agent,
    expected_output="Detailed blog post outline with sections and key points",
    context=[research_step]
)

# Step 4: Worker writes draft
draft_step = Task(
    description="SUBTASK 3: Using the outline, write a complete draft of the blog post. "
                "Make it engaging and informative (approximately 300 words).",
    agent=worker_agent,
    expected_output="Complete blog post draft",
    context=[outline_step]
)

# Step 5: Worker edits and finalizes
edit_step = Task(
    description="SUBTASK 4: Review and edit the draft. Improve clarity, fix any issues, "
                "and ensure the blog post is polished and ready for publication.",
    agent=worker_agent,
    expected_output="Final, polished blog post ready for publication",
    context=[draft_step]
)

# Create the orchestrator-worker crew
orchestrator_crew = Crew(
    agents=[manager_agent, worker_agent],
    tasks=[planning_task, research_step, outline_step, draft_step, edit_step],
    verbose=True,
    process=Process.sequential
)

# Execute the orchestrator-worker pattern
print("=" * 60)
print("PATTERN D: ORCHESTRATOR-WORKER EXAMPLE")
print("=" * 60)
print("Manager breaks down task → Worker executes each step")
print("=" * 60 + "\n")

result = await orchestrator_crew.kickoff_async(inputs={"topic": "The Future of Renewable Energy"})

print("\n" + "=" * 60)
print("FINAL BLOG POST:")
print("=" * 60)
print(result)

PATTERN D: ORCHESTRATOR-WORKER EXAMPLE
Manager breaks down task → Worker executes each step



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.44                                                                                       │
│  Latest version:  1.15.45                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e5f3857b-ba0f-4ccb-a592-6f90a60ef439                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a plan for writing a blog post about The Future of Renewable Energy. List the key steps needed    │
│  (research, outline, draft, edit).                                                                              │
│  ID: 5bceb358-0117-4a89-b12f-0ac0cf40ee1b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│  Task: Create a plan for writing a blog post about The Future of Renewable Energy. List the key steps needed    │
│  (research, outline, draft, edit).                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (20 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Worker                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # The Future of Renewable Energy: Trends, Challenges, and Opportunities Ahead                                  │
│                                                                                                                 │
│  Renewable energy is no longer a distant climate ambition—it is already reshaping the global power system.      │
│  Solar and wind are among the fastest-growing sources of electricity worldwide, and in many markets they are    │
│  becoming the default choice for new generation. But the future of renewable energy is about more than simply   │
│  adding more clean power. It is about building an integrated energy ecosystem that includes storage, smarter    │
│  grids, electrification, and supportive policy.                                                                 │
│                                                                                                                 │
│  ## Why Renewable Energy Is Growing So Quickly                                                                  │
│                                                                                                                 │
│  One of the biggest reasons renewables are advancing rapidly is cost. Over the past decade, solar PV, onshore   │
│  wind, and batteries have become dramatically cheaper, making clean power increasingly competitive with fossil  │
│  fuels. In many regions, new solar and wind projects are now among the least expensive sources of electricity   │
│  available.                                                                                                     │
│                                                                                                                 │
│  Solar is likely to remain a leading growth technology because it is scalable, fast to install, and flexible    │
│  enough to fit utility-scale farms, rooftops, and even floating systems. When paired with batteries, solar      │
│  becomes far more reliable, allowing electricity to flow even after sunset.                                     │
│                                                                                                                 │
│  Wind power will also remain essential. Onshore wind is already one of the lowest-cost electricity sources in   │
│  many regions, while offshore wind offers massive potential near coastal population centers. Hydropower will    │
│  continue to provide stability, and geothermal could become a much bigger player if drilling technology         │
│  continues to improve.                                                                                          │
│                                                                                                                 │
│  ## The Grid May Be the Biggest Challenge                                                                       │
│                                                                                                                 │
│  As renewable capacity grows, the biggest bottleneck may not be generation—it may be the grid. Transmission     │
│  lines, distribution systems, and storage capacity all need major upgrades to keep pace with clean energy       │
│  deployment.                                                                                                    │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: SUBTASK 4: Review and edit the draft. Improve clarity, fix any issues, and ensure the blog post is       │
│  polished and ready for publication.                                                                            │
│  Agent: Content Worker                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL BLOG POST:
# The Future of Renewable Energy: Trends, Challenges, and Opportunities Ahead

Renewable energy is no longer a distant climate ambition—it is already reshaping the global power system. Solar and wind are among the fastest-growing sources of electricity worldwide, and in many markets they are becoming the default choice for new generation. But the future of renewable energy is about more than simply adding more clean power. It is about building an integrated energy ecosystem that includes storage, smarter grids, electrification, and supportive policy.

## Why Renewable Energy Is Growing So Quickly

One of the biggest reasons renewables are advancing rapidly is cost. Over the past decade, solar PV, onshore wind, and batteries have become dramatically cheaper, making clean power increasingly competitive with fossil fuels. In many regions, new solar and wind projects are now among the least expensive sources of electricity available.

Solar is likely to remain a leading g

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e5f3857b-ba0f-4ccb-a592-6f90a60ef439                                                                       │
│  Final Output: # The Future of Renewable Energy: Trends, Challenges, and Opportunities Ahead                    │
│                                                                                                                 │
│  Renewable energy is no longer a distant climate ambition—it is already reshaping the global power system.      │
│  Solar and wind are among the fastest-growing sources of electricity worldwide, and in many markets they are    │
│  becoming the default choice for new generation. But the future of renewable energy is about more than simply   │
│  adding more clean power. It is about building an integrated energy ecosystem that includes storage, smarter    │
│  grids, electrification, and supportive policy.                                                                 │
│                                                                                                                 │
│  ## Why Renewable Energy Is Growing So Quickly                                                                  │
│                                                                                                                 │
│  One of the biggest reasons renewables are advancing rapidly is cost. Over the past decade, solar PV, onshore   │
│  wind, and batteries have become dramatically cheaper, making clean power increasingly competitive with fossil  │
│  fuels. In many regions, new solar and wind projects are now among the least expensive sources of electricity   │
│  available.                                                                                                     │
│                                                                                                                 │
│  Solar is likely to remain a leading growth technology because it is scalable, fast to install, and flexible    │
│  enough to fit utility-scale farms, rooftops, and even floating systems. When paired with batteries, solar      │
│  becomes far more reliable, allowing electricity to flow even after sunset.                                     │
│                                                                                                                 │
│  Wind power will also remain essential. Onshore wind is already one of the lowest-cost electricity sources in   │
│  many regions, while offshore wind offers massive potential near coastal population centers. Hydropower will    │
│  continue to provide stability, and geothermal could become a much bigger player if drilling technology         │
│  continues to improve.                                                                                          │
│                                                                                                                 │
│  ## The Grid May Be the Biggest Challenge                                                                       │
│                                                                                                                 │
│  As renewable capacity grows, the biggest bottleneck may not be generation—it may be the grid. Transmission     │
│  lines, distribution systems, and storage capacity all need major upgrades to keep pace with clean energy       │
│  deployment.                                                                                                    │
│                                                       

## 7. Pattern E: Evaluator-Optimizer

**Concept**: An iterative feedback loop where one agent generates content, another evaluates it, and the generator improves based on feedback.

**Use Case**: When output quality is critical and needs iterative refinement.

**Example**: Content Generator → Evaluator → Improved Generator (Feedback Loop)

In [8]:
# Generator Agent: Creates initial content
generator_agent = Agent(
    role="Content Generator",
    goal="Create compelling marketing copy for {product}",
    backstory="You are a creative copywriter who generates engaging marketing content. "
              "You're open to feedback and continuously improve your work.",
    llm=llm,
    verbose=True
)

# Evaluator Agent: Critiques the content
evaluator_agent = Agent(
    role="Marketing Copy Evaluator",
    goal="Evaluate marketing copy and provide constructive feedback",
    backstory="You are a seasoned marketing director with a keen eye for effective copy. "
              "You evaluate content based on: clarity, persuasiveness, emotional appeal, "
              "and call-to-action strength.",
    llm=llm,
    verbose=True
)

# Optimizer Agent: Improves based on feedback
optimizer_agent = Agent(
    role="Content Optimizer",
    goal="Improve marketing copy based on evaluator feedback",
    backstory="You are a skilled editor who excels at refining content based on feedback. "
              "You make targeted improvements while maintaining the original creative vision.",
    llm=llm,
    verbose=True
)

# Iteration 1: Generate initial content
generation_task = Task(
    description="Write a compelling 100-word marketing copy for {product}. "
                "Include a strong headline and a clear call-to-action.",
    agent=generator_agent,
    expected_output="Initial marketing copy with headline and call-to-action"
)

# Iteration 2: Evaluate the content
evaluation_task = Task(
    description="Evaluate the marketing copy. Rate it on:\n"
                "1. Clarity (1-10)\n"
                "2. Persuasiveness (1-10)\n"
                "3. Emotional Appeal (1-10)\n"
                "4. Call-to-Action Strength (1-10)\n\n"
                "Provide specific suggestions for improvement.",
    agent=evaluator_agent,
    expected_output="Detailed evaluation with ratings and improvement suggestions",
    context=[generation_task]
)

# Iteration 3: Optimize based on feedback
optimization_task = Task(
    description="Improve the marketing copy based on the evaluator's feedback. "
                "Address each suggestion and create a refined version.",
    agent=optimizer_agent,
    expected_output="Improved marketing copy incorporating feedback",
    context=[generation_task, evaluation_task]
)

# Iteration 4: Re-evaluate the improved version
final_evaluation_task = Task(
    description="Re-evaluate the improved marketing copy. Compare it to the original "
                "and confirm the improvements. Provide final ratings.",
    agent=evaluator_agent,
    expected_output="Final evaluation confirming improvements",
    context=[optimization_task]
)

# Create the evaluator-optimizer crew
feedback_loop_crew = Crew(
    agents=[generator_agent, evaluator_agent, optimizer_agent],
    tasks=[generation_task, evaluation_task, optimization_task, final_evaluation_task],
    verbose=True,
    process=Process.sequential
)

# Execute the feedback loop
print("=" * 60)
print("PATTERN E: EVALUATOR-OPTIMIZER EXAMPLE")
print("=" * 60)
print("Generate → Evaluate → Optimize → Re-evaluate")
print("=" * 60 + "\n")

result = await feedback_loop_crew.kickoff_async(
    inputs={"product": "An AI-powered personal finance app that helps users save money"}
)

print("\n" + "=" * 60)
print("FINAL RESULT (After Feedback Loop):")
print("=" * 60)
print(result)

PATTERN E: EVALUATOR-OPTIMIZER EXAMPLE
Generate → Evaluate → Optimize → Re-evaluate



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.44                                                                                       │
│  Latest version:  1.15.45                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 32a6ab8a-71bd-4273-a31a-ab47838ba01d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Write a compelling 100-word marketing copy for An AI-powered personal finance app that helps users save  │
│  money. Include a strong headline and a clear call-to-action.                                                   │
│  ID: a2be310f-901a-465d-9497-203ce6758222                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Generator                                                                                       │
│                                                                                                                 │
│  Task: Write a compelling 100-word marketing copy for An AI-powered personal finance app that helps users save  │
│  money. Include a strong headline and a clear call-to-action.                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (15 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Copy Evaluator                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Final Evaluation: Improved Copy Confirmed**                                                                  │
│                                                                                                                 │
│  The revised version is a clear improvement over the original. It does a better job of connecting the product   │
│  benefits to the reader’s pain points, and the messaging feels more polished, persuasive, and action-oriented.  │
│                                                                                                                 │
│  ### What Improved                                                                                              │
│  - **Clarity:** The core value is easier to understand right away. The copy clearly explains what the app       │
│  does: tracks spending, finds savings, gives insights, and helps build better habits.                           │
│  - **Persuasiveness:** The benefits are stronger and more concrete. Phrases like “finds hidden savings,”        │
│  “budget alerts,” and “recommend the next best step” make the product feel more useful and differentiated.      │
│  - **Emotional appeal:** The copy now taps into common money frustrations and aspirations, such as “Stop        │
│  wondering where your paycheck went” and “feel confident about your money again.” That makes it more            │
│  relatable.                                                                                                     │
│  - **Call to action:** The CTA is stronger and more motivating. “Start saving smarter today—download the app    │
│  now…” is direct, benefit-driven, and easier to act on.                                                         │
│                                                                                                                 │
│  ### Comparison to the Original                                                                                 │
│  Compared with the original, this version:                                                                      │
│  - Feels more user-focused                                                                                      │
│  - Explains the product more clearly                                                                            │
│  - Better highlights the AI advantage                                                                           │
│  - Creates more urgency and relevance                                                                           │
│  - Ends with a stronger conversion push                                                                         │
│                                                                                                                 │
│  ### Final Ratings                                                                                              │
│  - **Clarity:** 9/10                                                                                            │
│  - **Persuasiveness:** 9/10                                                                                     │
│  - **Emotional Appeal:** 8.5/10                                                                                 │
│  - **Call-to-Action Strength:** 9/10                                                                            │
│  - **Overall Effectiveness:** 9/10                     

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Re-evaluate the improved marketing copy. Compare it to the original and confirm the improvements.        │
│  Provide final ratings.                                                                                         │
│  Agent: Marketing Copy Evaluator                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL RESULT (After Feedback Loop):
**Final Evaluation: Improved Copy Confirmed**

The revised version is a clear improvement over the original. It does a better job of connecting the product benefits to the reader’s pain points, and the messaging feels more polished, persuasive, and action-oriented.

### What Improved
- **Clarity:** The core value is easier to understand right away. The copy clearly explains what the app does: tracks spending, finds savings, gives insights, and helps build better habits.
- **Persuasiveness:** The benefits are stronger and more concrete. Phrases like “finds hidden savings,” “budget alerts,” and “recommend the next best step” make the product feel more useful and differentiated.
- **Emotional appeal:** The copy now taps into common money frustrations and aspirations, such as “Stop wondering where your paycheck went” and “feel confident about your money again.” That makes it more relatable.
- **Call to action:** The CTA is stronger and more motivating. 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 32a6ab8a-71bd-4273-a31a-ab47838ba01d                                                                       │
│  Final Output: **Final Evaluation: Improved Copy Confirmed**                                                    │
│                                                                                                                 │
│  The revised version is a clear improvement over the original. It does a better job of connecting the product   │
│  benefits to the reader’s pain points, and the messaging feels more polished, persuasive, and action-oriented.  │
│                                                                                                                 │
│  ### What Improved                                                                                              │
│  - **Clarity:** The core value is easier to understand right away. The copy clearly explains what the app       │
│  does: tracks spending, finds savings, gives insights, and helps build better habits.                           │
│  - **Persuasiveness:** The benefits are stronger and more concrete. Phrases like “finds hidden savings,”        │
│  “budget alerts,” and “recommend the next best step” make the product feel more useful and differentiated.      │
│  - **Emotional appeal:** The copy now taps into common money frustrations and aspirations, such as “Stop        │
│  wondering where your paycheck went” and “feel confident about your money again.” That makes it more            │
│  relatable.                                                                                                     │
│  - **Call to action:** The CTA is stronger and more motivating. “Start saving smarter today—download the app    │
│  now…” is direct, benefit-driven, and easier to act on.                                                         │
│                                                                                                                 │
│  ### Comparison to the Original                                                                                 │
│  Compared with the original, this version:                                                                      │
│  - Feels more user-focused                                                                                      │
│  - Explains the product more clearly                                                                            │
│  - Better highlights the AI advantage                                                                           │
│  - Creates more urgency and relevance                                                                           │
│  - Ends with a stronger conversion push                                                                         │
│                                                                                                                 │
│  ### Final Ratings                                                                                              │
│  - **Clarity:** 9/10                                                                                            │
│  - **Persuasiveness:** 9/10                                                                                     │
│  - **Emotional Appeal:** 8.5/10                                                                                 │
│  - **Call-to-Action Strength:** 9/10                                                                            │
│  - **Overall Effectiveness:** 9/10                    

## 8. Conclusion: When to Use Each Pattern

Now that we've explored the five fundamental agentic patterns, let's summarize when to use each one:

### Pattern Selection Guide

| Pattern | Best Used When | Key Benefits | Example Use Cases |
|---------|---------------|--------------|-------------------|
| **Prompt Chaining** | Tasks naturally flow in sequence, each step depends on the previous | Simple, predictable, easy to debug | Data pipeline (extract → transform → load), Research → Summary |
| **Routing** | You have multiple specialized agents and need dynamic selection | Efficient resource use, specialized expertise | Customer service routing, Intent classification, Dynamic tool selection |
| **Parallelization** | Independent subtasks can run simultaneously | Time efficiency, concurrent processing | Market research + competitor analysis, Multiple data source aggregation |
| **Orchestrator-Worker** | Complex tasks need decomposition into managed steps | Clear structure, scalable delegation | Project planning, Multi-step content creation, Complex workflows |
| **Evaluator-Optimizer** | Quality is critical and iterative improvement needed | High-quality output, continuous refinement | Content creation, Code review, Strategic planning |

### Combining Patterns

These patterns aren't mutually exclusive! In production systems, you might combine them:
- **Routing + Parallelization**: Route queries to multiple specialists who work in parallel
- **Orchestrator-Worker + Evaluator-Optimizer**: Orchestrate a process with quality checks at each step
- **Prompt Chaining + Routing**: Chain multiple steps where each step routes to different specialists

### Key Takeaways

1. **Start Simple**: Begin with Prompt Chaining for linear workflows
2. **Add Intelligence**: Use Routing when you need dynamic decision-making
3. **Scale Performance**: Apply Parallelization for time-critical operations
4. **Manage Complexity**: Use Orchestrator-Worker for complex, multi-step tasks
5. **Ensure Quality**: Implement Evaluator-Optimizer when output quality is paramount

### Next Steps

- Experiment with each pattern using your own use cases
- Try combining patterns for more sophisticated workflows
- Explore CrewAI's additional features: custom tools, memory, and advanced configurations
- Monitor agent performance and iterate on your designs

Happy building with CrewAI! 🚀